In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Seed complete: transition seeding -> cdc and checkpoint V0 (captured BEFORE the seed).
# The next cdc cycle reads CHANGETABLE(CHANGES, V0) to reconcile rows changed during the seed.
from state_store import CDC

src_database = widget("src_database")
src_schema = widget("src_schema")
src_table = widget("src_table")
v0 = int(widget("v0"))

store.advance_version(src_database, src_schema, src_table, ct_version=v0, phase=CDC, detail="seed complete")
print("checkpointed seed V0 =", v0, "phase = cdc")